# 18 v8. Lightweight KSJ L03-b land-use diagnosis and safe ratio construction

この notebook は，18 v7 のように L03-b ファイルを一気に全読込せず，まず **ZIP 内部の診断** と **少数ファイルでの読み取り確認** を行い，読めることを確認した範囲だけで `paddy_ratio_5km`, `farmland_ratio_5km`, `forest_ratio_5km` などを作成する軽量版です．

主出力はこれまでと同じです．

```text
/content/drive/MyDrive/avian_influenza_project/processed/domain_features/grid_environment_features.csv
```

まずは `MAX_FILES_TO_PROCESS = 5` 程度で診断し，成功したら増やしてください．

In [13]:
# ============================================================
# 18 v8. Lightweight KSJ L03-b diagnosis and safe landuse ratios
# Version: 18_v8_lightweight_ksj_l03b_diagnosis_landuse_ratios
# ============================================================

NOTEBOOK_VERSION = "18_v8_lightweight_ksj_l03b_diagnosis_landuse_ratios"
print("NOTEBOOK VERSION:", NOTEBOOK_VERSION)

from pathlib import Path
import os, re, json, zipfile, shutil, warnings, math, subprocess, sys, importlib.util
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print("Drive mount skipped:", e)

PROC_DIR = Path('/content/drive/MyDrive/avian_influenza_project/processed')
MODEL_DIR = PROC_DIR / 'model_outputs_riskmap_eval'
DOMAIN_DIR = PROC_DIR / 'domain_features'
GIS_RAW_DIR = PROC_DIR / 'gis_raw'
KSJ_DIR = GIS_RAW_DIR / 'ksj_l03b'
UNZIP_DIR = PROC_DIR / 'gis_unzipped' / 'ksj_l03b_v8'

for d in [PROC_DIR, MODEL_DIR, DOMAIN_DIR, GIS_RAW_DIR, KSJ_DIR, UNZIP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ------------------------------
# Safety / speed settings
# ------------------------------
MAX_ZIPS_TO_INSPECT = 20       # ZIP内部一覧だけを見る上限
MAX_FILES_TO_DIAGNOSE = 3      # 実読込診断するファイル数
MAX_FILES_TO_PROCESS = 120       # 実際に土地利用点を作るファイル数．重ければ 3〜5 に下げる
MAX_ROWS_PER_FILE = 300000     # CSV等の読込上限，GMLでは効かない場合あり
USE_GPD_READ_FILE = True       # True: geopandasで読めるか試す
USE_XML_HEURISTIC = False      # 重いのでデフォルトFalse．必要時だけTrue

print("PROC_DIR:", PROC_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("DOMAIN_DIR:", DOMAIN_DIR)
print("KSJ_DIR:", KSJ_DIR)
print("UNZIP_DIR:", UNZIP_DIR)


NOTEBOOK VERSION: 18_v8_lightweight_ksj_l03b_diagnosis_landuse_ratios
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROC_DIR: /content/drive/MyDrive/avian_influenza_project/processed
MODEL_DIR: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval
DOMAIN_DIR: /content/drive/MyDrive/avian_influenza_project/processed/domain_features
KSJ_DIR: /content/drive/MyDrive/avian_influenza_project/processed/gis_raw/ksj_l03b
UNZIP_DIR: /content/drive/MyDrive/avian_influenza_project/processed/gis_unzipped/ksj_l03b_v8


## 1. 必要パッケージの準備

In [14]:
def ensure_package(pkg, import_name=None):
    import_name = import_name or pkg
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    else:
        print(f"{pkg} already available")

for pkg, imp in [("geopandas", "geopandas"), ("pyogrio", "pyogrio"), ("rtree", "rtree"), ("scikit-learn", "sklearn")]:
    try:
        ensure_package(pkg, imp)
    except Exception as e:
        print("Package install/check failed:", pkg, e)

import geopandas as gpd
from shapely.geometry import Point
from sklearn.neighbors import BallTree

print("Packages ready.")


geopandas already available
pyogrio already available
rtree already available
scikit-learn already available
Packages ready.


## 2. グリッドマスタと既存環境特徴量の読込

18 v3/v4/v5/v6で作った `grid_environment_features.csv` をベースに，土地利用比率だけを追加します．

In [15]:
# Load grid/environment feature base
base_candidates = [
    DOMAIN_DIR / 'grid_environment_features.csv',
    MODEL_DIR / '18_v6_grid_environment_features_for_14.csv',
    MODEL_DIR / '18_v5_grid_environment_features_for_14.csv',
    MODEL_DIR / '18_v3_grid_environment_features_for_14.csv',
    MODEL_DIR / '15_grid_master_for_domain_features.csv',
]

domain_features = None
base_source = None
for p in base_candidates:
    if p.exists():
        try:
            tmp = pd.read_csv(p)
            if {'grid_id', 'grid_lat', 'grid_lon'}.issubset(tmp.columns):
                domain_features = tmp.copy()
                base_source = p
                break
        except Exception as e:
            print("failed:", p, e)

if domain_features is None:
    raise FileNotFoundError("grid_id, grid_lat, grid_lon を含むベースファイルが見つかりません．")

# Normalize coordinate columns
domain_features['grid_lat'] = pd.to_numeric(domain_features['grid_lat'], errors='coerce')
domain_features['grid_lon'] = pd.to_numeric(domain_features['grid_lon'], errors='coerce')
domain_features = domain_features.dropna(subset=['grid_lat', 'grid_lon']).drop_duplicates('grid_id').reset_index(drop=True)

grid_master = domain_features[['grid_id', 'grid_lat', 'grid_lon']].copy()
print("base_source:", base_source)
print("domain_features shape:", domain_features.shape)
display(domain_features.head())


base_source: /content/drive/MyDrive/avian_influenza_project/processed/domain_features/grid_environment_features.csv
domain_features shape: (5491, 22)


,grid_id,grid_lat,grid_lon,dist_to_coast_km,dist_to_river_km,dist_to_waterbody_km,dist_to_lake_or_reservoir_km,elevation_m,slope_deg,dist_to_wetland_km,...,dist_to_migratory_bird_site_km,dist_to_wildbird_surveillance_site_km,wildbird_site_count_30km,dist_to_poultry_farm_km,landuse_sample_count_5km,paddy_ratio_5km,farmland_ratio_5km,forest_ratio_5km,urban_ratio_5km,waterbody_ratio_5km
0,G000015,24.250542,123.786000,2.539137,598.279685,924.442013,924.442013,0.0,0.000000,NaN,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
1,G000016,24.332420,123.786000,4.140221,590.298566,915.326001,915.326001,335.0,7.930704,NaN,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
2,G000033,24.414246,124.235160,2.276512,614.386971,928.204439,928.204439,26.0,0.336454,NaN,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
3,G000048,24.822575,125.313140,0.518983,668.111228,947.981371,947.981371,41.0,1.251468,NaN,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
4,G000080,26.200762,127.738594,3.340534,833.159624,1007.796823,1007.796823,24.0,0.117843,NaN,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN


## 3. 既存の L03-b ZIP を探す

18 v6で落としたZIPを優先して使います．なければ `processed/gis_raw/ksj_l03b` や `processed/gis_raw` を探します．

In [16]:
# Find L03-b zip files
zip_candidates = []
search_dirs = [KSJ_DIR, GIS_RAW_DIR, PROC_DIR / 'gis_downloads', PROC_DIR / 'gis_unzipped']
for d in search_dirs:
    if d.exists():
        for p in d.rglob('*.zip'):
            name = p.name.lower()
            if ('l03' in name or 'landuse' in name or '土地利用' in name) and ('l03-b' in name or 'l03b' in name or 'l03' in name):
                zip_candidates.append(p)

# Also use v6 selected list if available
sel_path = MODEL_DIR / '18_v6_l03b_selected_zip_files.csv'
if sel_path.exists():
    try:
        sel = pd.read_csv(sel_path)
        if 'zip_path' in sel.columns:
            for z in sel['zip_path'].dropna().astype(str):
                p = Path(z)
                if p.exists() and p.suffix.lower() == '.zip':
                    zip_candidates.append(p)
    except Exception as e:
        print("Could not read v6 selected list:", e)

zip_candidates = sorted(set(zip_candidates), key=lambda x: str(x))
zip_inv = pd.DataFrame([{
    'zip_path': str(p),
    'name': p.name,
    'size_mb': p.stat().st_size/1024/1024,
} for p in zip_candidates])
zip_inv.to_csv(MODEL_DIR / '18_v8_l03b_zip_candidates.csv', index=False, encoding='utf-8-sig')
print("ZIP candidates:", len(zip_candidates))
display(zip_inv.head(30))


ZIP candidates: 135


,zip_path,name,size_mb
0,/content/drive/MyDrive/avian_influenza_project...,L03-b-21_3623-jgd2011_GML.zip,3.685064
1,/content/drive/MyDrive/avian_influenza_project...,L03-b-21_3624-jgd2011_GML.zip,4.355383
2,/content/drive/MyDrive/avian_influenza_project...,L03-b-21_3725-jgd2011_GML.zip,3.693626
3,/content/drive/MyDrive/avian_influenza_project...,L03-b-21_3927-jgd2011_GML.zip,11.383656
4,/content/drive/MyDrive/avian_influenza_project...,L03-b-21_3928-jgd2011_GML.zip,2.353333
5,/content/drive/MyDrive/avian_influenza_project...,L03-b-21_4027-jgd2011_GML.zip,2.354659
6,/content/drive/MyDrive/avian_influenza_project...,L03-b-21_4028-jgd2011_GML.zip,3.678205
7,/content/drive/MyDrive/avian_influenza_project...,L03-b-21_4128-jgd2011_GML.zip,3.029935
8,/content/drive/MyDrive/avian_influenza_project...,L03-b-21_4129-jgd2011_GML.zip,1.020139
9,/content/drive/MyDrive/avian_influenza_project...,L03-b-21_4229-jgd2011_GML.zip,7.020871


## 4. ZIP内部を診断する

ここではZIPを読まず，内部ファイル名だけを一覧化します．`no_gml` 問題の原因確認用です．

In [17]:
member_rows = []
for zp in zip_candidates[:MAX_ZIPS_TO_INSPECT]:
    try:
        with zipfile.ZipFile(zp, 'r') as z:
            names = z.namelist()
        for n in names[:200]:
            member_rows.append({
                'zip_path': str(zp),
                'member': n,
                'suffix': Path(n).suffix.lower(),
                'is_candidate_data': Path(n).suffix.lower() in ['.gml', '.xml', '.geojson', '.json', '.shp', '.gpkg', '.csv']
            })
    except Exception as e:
        member_rows.append({'zip_path': str(zp), 'member': '', 'suffix': '', 'is_candidate_data': False, 'error': str(e)})

members = pd.DataFrame(member_rows)
members.to_csv(MODEL_DIR / '18_v8_l03b_zip_member_inventory.csv', index=False, encoding='utf-8-sig')
print("member rows:", len(members))
display(members.head(50))
display(members.groupby('suffix').size().reset_index(name='n').sort_values('n', ascending=False).head(20) if not members.empty else members)


member rows: 140


,zip_path,member,suffix,is_candidate_data
0,/content/drive/MyDrive/avian_influenza_project...,KS-META-L03-b-21_3623.xml,.xml,True
1,/content/drive/MyDrive/avian_influenza_project...,L03-b-21_3623.dbf,.dbf,False
2,/content/drive/MyDrive/avian_influenza_project...,L03-b-21_3623.geojson,.geojson,True
3,/content/drive/MyDrive/avian_influenza_project...,L03-b-21_3623.prj,.prj,False
4,/content/drive/MyDrive/avian_influenza_project...,L03-b-21_3623.shp,.shp,True
5,/content/drive/MyDrive/avian_influenza_project...,L03-b-21_3623.shx,.shx,False
6,/content/drive/MyDrive/avian_influenza_project...,L03-b-21_3623.xml,.xml,True
7,/content/drive/MyDrive/avian_influenza_project...,KS-META-L03-b-21_3624.xml,.xml,True
8,/content/drive/MyDrive/avian_influenza_project...,L03-b-21_3624.dbf,.dbf,False
9,/content/drive/MyDrive/avian_influenza_project...,L03-b-21_3624.geojson,.geojson,True


,suffix,n
5,.xml,40
0,.dbf,20
1,.geojson,20
2,.prj,20
3,.shp,20
4,.shx,20


## 5. ZIPを展開してデータファイル候補を作る

全ZIPを一気に処理せず，まず候補ファイル一覧を作ります．

In [18]:
extract_logs = []
for zp in zip_candidates:
    out_dir = UNZIP_DIR / zp.stem
    try:
        out_dir.mkdir(parents=True, exist_ok=True)
        # Extract only once
        if not any(out_dir.rglob('*')):
            with zipfile.ZipFile(zp, 'r') as z:
                z.extractall(out_dir)
        extract_logs.append({'zip_path': str(zp), 'out_dir': str(out_dir), 'status': 'extracted_or_cached'})
    except Exception as e:
        extract_logs.append({'zip_path': str(zp), 'out_dir': str(out_dir), 'status': 'failed', 'error': str(e)})

pd.DataFrame(extract_logs).to_csv(MODEL_DIR / '18_v8_l03b_extract_log.csv', index=False, encoding='utf-8-sig')

suffixes = ['.gml', '.xml', '.geojson', '.json', '.shp', '.gpkg', '.csv']
data_files = []
for p in UNZIP_DIR.rglob('*'):
    if p.is_file() and p.suffix.lower() in suffixes:
        data_files.append(p)

data_files = sorted(set(data_files), key=lambda x: str(x))
file_inv = pd.DataFrame([{
    'path': str(p),
    'name': p.name,
    'suffix': p.suffix.lower(),
    'size_mb': p.stat().st_size/1024/1024
} for p in data_files])
file_inv.to_csv(MODEL_DIR / '18_v8_l03b_extracted_data_file_inventory.csv', index=False, encoding='utf-8-sig')
print("data files:", len(data_files))
display(file_inv.head(50))


data files: 540


,path,name,suffix,size_mb
0,/content/drive/MyDrive/avian_influenza_project...,KS-META-L03-b-21_3623.xml,.xml,0.013749
1,/content/drive/MyDrive/avian_influenza_project...,L03-b-21_3623.geojson,.geojson,36.254823
2,/content/drive/MyDrive/avian_influenza_project...,L03-b-21_3623.shp,.shp,14.267063
3,/content/drive/MyDrive/avian_influenza_project...,L03-b-21_3623.xml,.xml,8.550904
4,/content/drive/MyDrive/avian_influenza_project...,KS-META-L03-b-21_3624.xml,.xml,0.013749
5,/content/drive/MyDrive/avian_influenza_project...,L03-b-21_3624.geojson,.geojson,42.848432
6,/content/drive/MyDrive/avian_influenza_project...,L03-b-21_3624.shp,.shp,16.861057
7,/content/drive/MyDrive/avian_influenza_project...,L03-b-21_3624.xml,.xml,8.550904
8,/content/drive/MyDrive/avian_influenza_project...,KS-META-L03-b-21_3725.xml,.xml,0.013762
9,/content/drive/MyDrive/avian_influenza_project...,L03-b-21_3725.geojson,.geojson,36.255109


## 6. 土地利用コード判定関数

L03-bの列名やコード表が環境により異なる場合に備え，列名・値の両方から土地利用列を推定します．

In [19]:
PADDY_CODES = {'0100', '100', '1', '田', 'paddy', 'rice'}
FARMLAND_CODES = {'0200', '200', '2', 'その他の農用地', '農用地', 'farmland', 'agricultural'}
FOREST_CODES = {'0500', '500', '5', '森林', 'forest'}
URBAN_CODES = {'0700', '700', '7', '建物用地', '高層建物', '低層建物', '低層建物（密集地）', '工場', '公共施設等用地', '道路', '鉄道', 'urban', 'building', 'built'}
WATER_CODES = {'0900', '1000', '1100', '900', '9', '10', '11', '湖沼', '河川', '河川地及び湖沼', '海水域', 'water', 'lake', 'river'}

def normalize_code(x):
    if pd.isna(x):
        return ''
    s = str(x).strip()
    if re.fullmatch(r'\d+\.0', s):
        s = s[:-2]
    return s

def classify_landuse(x):
    s = normalize_code(x)
    sl = s.lower()
    if s in PADDY_CODES or sl in PADDY_CODES:
        return 'paddy'
    if s in FARMLAND_CODES or sl in FARMLAND_CODES:
        return 'farmland'
    if s in FOREST_CODES or sl in FOREST_CODES:
        return 'forest'
    if s in URBAN_CODES or sl in URBAN_CODES:
        return 'urban'
    if s in WATER_CODES or sl in WATER_CODES:
        return 'water'
    if s == '田' or '水田' in s:
        return 'paddy'
    if '農' in s or '畑' in s:
        return 'farmland'
    if '森林' in s or '山林' in s:
        return 'forest'
    if '建物' in s or '市街' in s or '都市' in s or '道路' in s or '鉄道' in s:
        return 'urban'
    if '湖' in s or '河川' in s or '水域' in s or '海水' in s:
        return 'water'
    return 'other'

def find_landuse_col(df):
    candidates = []
    for c in df.columns:
        if c == 'geometry':
            continue
        lc = str(c).lower()
        score = 0
        if any(k in lc for k in ['land', 'use', 'class', 'category', '土地', '利用', '種別', '区分', 'code', 'コード', 'l03', 'lu']):
            score += 5
        sample = df[c].dropna().astype(str).head(500)
        if not sample.empty:
            mapped = sample.map(classify_landuse)
            score += float((mapped != 'other').mean()) * 10
        if score > 0:
            candidates.append((score, c))
    if not candidates:
        return None
    return sorted(candidates, reverse=True)[0][1]

print('landuse functions ready')


landuse functions ready


## 7. 少数ファイルだけ読込診断

ここで列名と土地利用列推定を確認します．重ければセルを停止して `MAX_FILES_TO_DIAGNOSE` を下げてください．

In [20]:
diagnose_logs = []
for p in data_files[:MAX_FILES_TO_DIAGNOSE]:
    try:
        info = {'path': str(p), 'suffix': p.suffix.lower(), 'size_mb': p.stat().st_size/1024/1024}
        if p.suffix.lower() == '.csv':
            df = pd.read_csv(p, nrows=5000)
            info['status'] = 'csv_read'
            info['columns'] = '|'.join(map(str, df.columns[:50]))
            info['landuse_col_guess'] = find_landuse_col(df)
            info['n_preview_rows'] = len(df)
        elif p.suffix.lower() in ['.shp', '.geojson', '.json', '.gpkg', '.gml', '.xml'] and USE_GPD_READ_FILE:
            gdf = gpd.read_file(p, rows=5000)
            info['status'] = 'gpd_read_preview'
            info['columns'] = '|'.join(map(str, gdf.columns[:50]))
            info['landuse_col_guess'] = find_landuse_col(gdf.drop(columns='geometry', errors='ignore'))
            info['n_preview_rows'] = len(gdf)
            info['crs'] = str(gdf.crs)
        else:
            info['status'] = 'skipped_suffix_or_gpd_disabled'
        diagnose_logs.append(info)
    except Exception as e:
        diagnose_logs.append({'path': str(p), 'suffix': p.suffix.lower(), 'status': 'failed', 'error': str(e)[:500]})

diag = pd.DataFrame(diagnose_logs)
diag.to_csv(MODEL_DIR / '18_v8_l03b_read_diagnosis_log.csv', index=False, encoding='utf-8-sig')
display(diag)


,path,suffix,status,error,size_mb,columns,landuse_col_guess,n_preview_rows,crs
0,/content/drive/MyDrive/avian_influenza_project...,.xml,failed,'/content/drive/MyDrive/avian_influenza_projec...,NaN,NaN,NaN,NaN,NaN
1,/content/drive/MyDrive/avian_influenza_project...,.geojson,gpd_read_preview,NaN,36.254823,細分メッシュコード|土地利用種別|衛星写真撮影年月日|geometry,細分メッシュコード,5000.0,EPSG:6668
2,/content/drive/MyDrive/avian_influenza_project...,.shp,gpd_read_preview,NaN,14.267063,L03b_001|L03b_002|L03b_003|geometry,L03b_003,5000.0,EPSG:6668


## 8. 読めるファイルだけから土地利用サンプル点を作る

`MAX_FILES_TO_PROCESS` の範囲で処理します．成功したら上限を増やせます．

In [21]:
def read_landuse_points(path: Path):
    """Return DataFrame with lat, lon, landuse_group from one data file."""
    suffix = path.suffix.lower()
    if suffix == '.csv':
        df = pd.read_csv(path)
        # coordinate columns
        lat_col = next((c for c in df.columns if str(c).lower() in ['lat','latitude','緯度'] or 'lat' in str(c).lower()), None)
        lon_col = next((c for c in df.columns if str(c).lower() in ['lon','lng','longitude','経度'] or 'lon' in str(c).lower() or 'lng' in str(c).lower()), None)
        lu_col = find_landuse_col(df)
        if not (lat_col and lon_col and lu_col):
            return pd.DataFrame(), 'csv_no_lat_lon_or_landuse'
        out = df[[lat_col, lon_col, lu_col]].rename(columns={lat_col:'lat', lon_col:'lon', lu_col:'landuse_raw'})
    elif suffix in ['.shp', '.geojson', '.json', '.gpkg', '.gml', '.xml'] and USE_GPD_READ_FILE:
        gdf = gpd.read_file(path)
        if gdf.empty:
            return pd.DataFrame(), 'empty_gdf'
        if gdf.crs is None:
            gdf = gdf.set_crs('EPSG:4326', allow_override=True)
        try:
            gdf = gdf.to_crs('EPSG:4326')
        except Exception:
            pass
        lu_col = find_landuse_col(gdf.drop(columns='geometry', errors='ignore'))
        if lu_col is None:
            return pd.DataFrame(), 'no_landuse_col'
        # representative point for polygon; centroid-ish for points/lines
        try:
            reps = gdf.geometry.representative_point()
        except Exception:
            reps = gdf.geometry.centroid
        out = pd.DataFrame({'lat': reps.y.values, 'lon': reps.x.values, 'landuse_raw': gdf[lu_col].values})
    else:
        return pd.DataFrame(), 'unsupported'
    out['lat'] = pd.to_numeric(out['lat'], errors='coerce')
    out['lon'] = pd.to_numeric(out['lon'], errors='coerce')
    out = out.dropna(subset=['lat','lon'])
    out = out[(out['lat'].between(20, 47)) & (out['lon'].between(122, 154))]
    if out.empty:
        return pd.DataFrame(), 'no_japan_points'
    out['landuse_group'] = out['landuse_raw'].map(classify_landuse)
    out = out[out['landuse_group'].isin(['paddy','farmland','forest','urban','water'])]
    if out.empty:
        return pd.DataFrame(), 'no_mapped_landuse_points'
    out['source_file'] = str(path)
    return out[['lat','lon','landuse_group','source_file']], 'ok'

process_logs = []
point_frames = []
processed_count = 0

# Prefer files that diagnosis says are readable / have landuse col.
preferred = []
if (MODEL_DIR / '18_v8_l03b_read_diagnosis_log.csv').exists():
    diag = pd.read_csv(MODEL_DIR / '18_v8_l03b_read_diagnosis_log.csv')
    ok_paths = diag[(diag.get('status','').astype(str).str.contains('read', na=False)) & (diag.get('landuse_col_guess','').notna())]['path'].tolist()
    preferred = [Path(x) for x in ok_paths if Path(x).exists()]

ordered_files = []
for p in preferred + data_files:
    if p not in ordered_files:
        ordered_files.append(p)

for p in ordered_files:
    if processed_count >= MAX_FILES_TO_PROCESS:
        break
    try:
        pts, status = read_landuse_points(p)
        process_logs.append({'path': str(p), 'status': status, 'rows': len(pts), 'counts': json.dumps(pts['landuse_group'].value_counts().to_dict(), ensure_ascii=False) if not pts.empty else ''})
        if not pts.empty:
            point_frames.append(pts)
            processed_count += 1
    except Exception as e:
        process_logs.append({'path': str(p), 'status': 'failed', 'error': str(e)[:500]})

process_log_df = pd.DataFrame(process_logs)
process_log_df.to_csv(MODEL_DIR / '18_v8_l03b_process_log.csv', index=False, encoding='utf-8-sig')
print('non-empty processed files:', len(point_frames))
display(process_log_df.head(50))


non-empty processed files: 120


,path,status,rows,counts,error
0,/content/drive/MyDrive/avian_influenza_project...,no_mapped_landuse_points,0.0,,NaN
1,/content/drive/MyDrive/avian_influenza_project...,no_mapped_landuse_points,0.0,,NaN
2,/content/drive/MyDrive/avian_influenza_project...,failed,NaN,NaN,'/content/drive/MyDrive/avian_influenza_projec...
3,/content/drive/MyDrive/avian_influenza_project...,failed,NaN,NaN,index 0 is out of bounds for axis 0 with size 0
4,/content/drive/MyDrive/avian_influenza_project...,failed,NaN,NaN,'/content/drive/MyDrive/avian_influenza_projec...
5,/content/drive/MyDrive/avian_influenza_project...,no_mapped_landuse_points,0.0,,NaN
6,/content/drive/MyDrive/avian_influenza_project...,no_mapped_landuse_points,0.0,,NaN
7,/content/drive/MyDrive/avian_influenza_project...,failed,NaN,NaN,index 0 is out of bounds for axis 0 with size 0
8,/content/drive/MyDrive/avian_influenza_project...,failed,NaN,NaN,'/content/drive/MyDrive/avian_influenza_projec...
9,/content/drive/MyDrive/avian_influenza_project...,no_mapped_landuse_points,0.0,,NaN


## 9. 5km圏内土地利用比率を作る

土地利用点が作れた場合のみ，`paddy_ratio_5km` などを更新します．

In [22]:
def add_landuse_ratios(points_df, grid_df, radius_km=5.0):
    dst = np.deg2rad(np.c_[points_df['lat'].values, points_df['lon'].values])
    src = np.deg2rad(np.c_[grid_df['grid_lat'].values, grid_df['grid_lon'].values])
    tree = BallTree(dst, metric='haversine')
    neighbors = tree.query_radius(src, r=radius_km/6371.0088)
    labels = points_df['landuse_group'].values
    groups = ['paddy','farmland','forest','urban','water']
    result = pd.DataFrame({'grid_id': grid_df['grid_id'].values})
    counts = []
    for inds in neighbors:
        counts.append(len(inds))
    result['landuse_sample_count_5km'] = counts
    for g in groups:
        vals = []
        for inds in neighbors:
            if len(inds) == 0:
                vals.append(np.nan)
            else:
                vals.append(float(np.mean(labels[inds] == g)))
        col = {'paddy':'paddy_ratio_5km','farmland':'farmland_ratio_5km','forest':'forest_ratio_5km','urban':'urban_ratio_5km','water':'waterbody_ratio_5km'}[g]
        result[col] = vals
    return result

landuse_creation_log = []
if point_frames:
    all_points = pd.concat(point_frames, ignore_index=True)
    # Drop exact duplicates to keep memory manageable
    all_points = all_points.drop_duplicates(subset=['lat','lon','landuse_group']).reset_index(drop=True)
    all_points.to_parquet(MODEL_DIR / '18_v8_l03b_landuse_points_sample.parquet', index=False)
    print('landuse points:', all_points.shape)
    display(all_points['landuse_group'].value_counts())
    ratios = add_landuse_ratios(all_points, grid_master, radius_km=5.0)
    for col in ['paddy_ratio_5km','farmland_ratio_5km','forest_ratio_5km','urban_ratio_5km','waterbody_ratio_5km','landuse_sample_count_5km']:
        if col in domain_features.columns:
            domain_features = domain_features.drop(columns=[col])
    domain_features = domain_features.merge(ratios, on='grid_id', how='left')
    landuse_creation_log.append({'status': 'created', 'n_points': len(all_points), 'n_grids_with_samples': int((ratios['landuse_sample_count_5km'] > 0).sum())})
else:
    landuse_creation_log.append({'status': 'no_landuse_points', 'message': 'No readable L03-b landuse points were created. Check diagnosis/process logs.'})

pd.DataFrame(landuse_creation_log).to_csv(MODEL_DIR / '18_v8_landuse_ratio_creation_log.csv', index=False, encoding='utf-8-sig')
display(pd.DataFrame(landuse_creation_log))


landuse points: (48537747, 4)


,count
landuse_group,
forest,34746938
paddy,4462624
urban,3669057
farmland,3519623
water,2139505


,status,n_points,n_grids_with_samples
0,created,48537747,4095


## 10. 保存と検証

In [23]:
target_cols = [
    'dist_to_coast_km','dist_to_river_km','dist_to_waterbody_km','dist_to_lake_or_reservoir_km',
    'waterbody_ratio_5km','urban_ratio_5km','elevation_m','slope_deg',
    'dist_to_wetland_km','paddy_ratio_5km','farmland_ratio_5km','forest_ratio_5km',
    'poultry_farm_density_10km','poultry_farms_in_grid','dist_to_migratory_bird_site_km',
    'dist_to_wildbird_surveillance_site_km','wildbird_site_count_30km','dist_to_poultry_farm_km',
    'landuse_sample_count_5km'
]
for c in target_cols:
    if c not in domain_features.columns:
        domain_features[c] = np.nan

# Remove forbidden columns
FORBIDDEN_PATTERNS = [r'^y_lead', r'target', r'label', r'outbreak', r'pred', r'risk', r'rank', r'num_birds', r'same_grid_past', r'neighbor_outbreak', r'lag_outbreak', r'rolling_outbreak']
def is_forbidden_col(c):
    s = str(c).lower()
    return any(re.search(pat, s) for pat in FORBIDDEN_PATTERNS)
forbidden_cols = [c for c in domain_features.columns if c != 'grid_id' and is_forbidden_col(c)]
if forbidden_cols:
    domain_features = domain_features.drop(columns=forbidden_cols)

keep_cols = ['grid_id','grid_lat','grid_lon']
numeric_cols = [c for c in domain_features.columns if c not in keep_cols and pd.api.types.is_numeric_dtype(domain_features[c])]
domain_features = domain_features[keep_cols + numeric_cols].drop_duplicates('grid_id')
feature_cols = [c for c in domain_features.columns if c not in keep_cols]

created_list = pd.DataFrame([{
    'column': c,
    'created': bool(domain_features[c].notna().any()),
    'non_null': int(domain_features[c].notna().sum()),
    'missing_rate': float(domain_features[c].isna().mean())
} for c in feature_cols])

column_report = pd.DataFrame([{
    'column': c,
    'created': bool(domain_features[c].notna().any()),
    'non_null': int(domain_features[c].notna().sum()),
    'missing_rate': float(domain_features[c].isna().mean()),
    'min': float(pd.to_numeric(domain_features[c], errors='coerce').min()) if pd.to_numeric(domain_features[c], errors='coerce').notna().any() else np.nan,
    'max': float(pd.to_numeric(domain_features[c], errors='coerce').max()) if pd.to_numeric(domain_features[c], errors='coerce').notna().any() else np.nan,
    'mean': float(pd.to_numeric(domain_features[c], errors='coerce').mean()) if pd.to_numeric(domain_features[c], errors='coerce').notna().any() else np.nan,
} for c in feature_cols])

validation = pd.DataFrame({
    'metric': ['n_rows','n_unique_grid_id','n_feature_cols_excluding_grid_lat_lon','n_forbidden_cols_removed'],
    'value': [len(domain_features), domain_features['grid_id'].nunique(), len(feature_cols), len(forbidden_cols)]
})

# Save final for notebook 14 v2
main_csv = DOMAIN_DIR / 'grid_environment_features.csv'
main_parquet = DOMAIN_DIR / 'grid_environment_features.parquet'
check_csv = MODEL_DIR / '18_v8_grid_environment_features_for_14.csv'
check_parquet = MODEL_DIR / '18_v8_grid_environment_features_for_14.parquet'

domain_features.to_csv(main_csv, index=False, encoding='utf-8-sig')
domain_features.to_parquet(main_parquet, index=False)
domain_features.to_csv(check_csv, index=False, encoding='utf-8-sig')
domain_features.to_parquet(check_parquet, index=False)
created_list.to_csv(MODEL_DIR / '18_v8_created_domain_feature_list.csv', index=False, encoding='utf-8-sig')
column_report.to_csv(MODEL_DIR / '18_v8_domain_feature_column_report.csv', index=False, encoding='utf-8-sig')
validation.to_csv(MODEL_DIR / '18_v8_domain_feature_validation_report.csv', index=False, encoding='utf-8-sig')
pd.DataFrame({'forbidden_columns_removed': forbidden_cols}).to_csv(MODEL_DIR / '18_v8_forbidden_columns_detected.csv', index=False, encoding='utf-8-sig')

print('Created feature columns with non-null values:')
display(created_list[created_list['created'] == True])
print('Saved:', main_csv)
display(validation)


Created feature columns with non-null values:


,column,created,non_null,missing_rate
0,dist_to_coast_km,True,5491,0.000000
1,dist_to_river_km,True,5491,0.000000
2,dist_to_waterbody_km,True,5491,0.000000
3,dist_to_lake_or_reservoir_km,True,5491,0.000000
4,elevation_m,True,5485,0.001093
5,slope_deg,True,5476,0.002732
13,landuse_sample_count_5km,True,5491,0.000000
14,paddy_ratio_5km,True,4095,0.254234
15,farmland_ratio_5km,True,4095,0.254234
16,forest_ratio_5km,True,4095,0.254234


Saved: /content/drive/MyDrive/avian_influenza_project/processed/domain_features/grid_environment_features.csv


,metric,value
0,n_rows,5491
1,n_unique_grid_id,5491
2,n_feature_cols_excluding_grid_lat_lon,19
3,n_forbidden_cols_removed,0


## 11. サマリーと保存確認

In [24]:
created_nonnull = created_list[created_list['created'] == True]['column'].tolist()
not_created = created_list[created_list['created'] == False]['column'].tolist()

report = []
report.append('# 18 v8 Summary: lightweight KSJ L03-b land-use diagnosis and ratio construction')
report.append('')
report.append(f'Generated by `{NOTEBOOK_VERSION}`.')
report.append('')
report.append('## Settings')
report.append(f'- MAX_ZIPS_TO_INSPECT: {MAX_ZIPS_TO_INSPECT}')
report.append(f'- MAX_FILES_TO_DIAGNOSE: {MAX_FILES_TO_DIAGNOSE}')
report.append(f'- MAX_FILES_TO_PROCESS: {MAX_FILES_TO_PROCESS}')
report.append(f'- USE_XML_HEURISTIC: {USE_XML_HEURISTIC}')
report.append('')
report.append('## Created features')
report.append(f'- Number of created feature columns excluding `grid_id`, `grid_lat`, `grid_lon`: {len(created_nonnull)}')
for c in created_nonnull:
    non_null = int(created_list.loc[created_list.column == c, 'non_null'].iloc[0])
    report.append(f'- `{c}`: non-null {non_null}')
report.append('')
report.append('## Not created')
for c in not_created:
    report.append(f'- `{c}`')
report.append('')
report.append('## Next step')
report.append('- If paddy/farmland/forest were created, rerun notebook 14 v2.')
report.append('- If not, inspect `18_v8_l03b_zip_member_inventory.csv`, `18_v8_l03b_read_diagnosis_log.csv`, and `18_v8_l03b_process_log.csv`.')

summary_path = MODEL_DIR / '18_v8_summary_report_lightweight_ksj_l03b_landuse_ratios.md'
summary_path.write_text('\n'.join(report), encoding='utf-8')

expected = [
    DOMAIN_DIR / 'grid_environment_features.csv',
    DOMAIN_DIR / 'grid_environment_features.parquet',
    MODEL_DIR / '18_v8_grid_environment_features_for_14.csv',
    MODEL_DIR / '18_v8_created_domain_feature_list.csv',
    MODEL_DIR / '18_v8_l03b_zip_member_inventory.csv',
    MODEL_DIR / '18_v8_l03b_read_diagnosis_log.csv',
    MODEL_DIR / '18_v8_l03b_process_log.csv',
    MODEL_DIR / '18_v8_landuse_ratio_creation_log.csv',
    MODEL_DIR / '18_v8_summary_report_lightweight_ksj_l03b_landuse_ratios.md',
]
saved = pd.DataFrame([{
    'file': p.name,
    'exists': p.exists(),
    'size_bytes': p.stat().st_size if p.exists() else 0,
    'path': str(p)
} for p in expected])
saved.to_csv(MODEL_DIR / '18_v8_saved_file_check.csv', index=False, encoding='utf-8-sig')

print(summary_path)
display(saved)
print('Done. If landuse ratios were created, rerun notebook 14 v2 next.')


/content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/18_v8_summary_report_lightweight_ksj_l03b_landuse_ratios.md


,file,exists,size_bytes,path
0,grid_environment_features.csv,True,1133608,/content/drive/MyDrive/avian_influenza_project...
1,grid_environment_features.parquet,True,524903,/content/drive/MyDrive/avian_influenza_project...
2,18_v8_grid_environment_features_for_14.csv,True,1133608,/content/drive/MyDrive/avian_influenza_project...
3,18_v8_created_domain_feature_list.csv,True,806,/content/drive/MyDrive/avian_influenza_project...
4,18_v8_l03b_zip_member_inventory.csv,True,19464,/content/drive/MyDrive/avian_influenza_project...
5,18_v8_l03b_read_diagnosis_log.csv,True,1074,/content/drive/MyDrive/avian_influenza_project...
6,18_v8_l03b_process_log.csv,True,122556,/content/drive/MyDrive/avian_influenza_project...
7,18_v8_landuse_ratio_creation_log.csv,True,62,/content/drive/MyDrive/avian_influenza_project...
8,18_v8_summary_report_lightweight_ksj_l03b_land...,True,1256,/content/drive/MyDrive/avian_influenza_project...


Done. If landuse ratios were created, rerun notebook 14 v2 next.
